# Práctica 6. Clasificación de datos con KNN

## Objetivo

Construir y evaluar un modelo de **K Vecinos Más Cercanos (KNN)** para clasificar el tipo de interacción de una reseña. Se retoma la lógica de las prácticas anteriores: una observación pertenece a **Calificación y texto** cuando tiene una calificación válida y texto; pertenece a **Solo texto** cuando conserva texto, pero no tiene calificación válida.

La variable `stars` no se utilizará como predictor porque está directamente relacionada con la existencia de una calificación y produciría fuga de información.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUTA_DATOS = "dataset_limpio.csv"
df = pd.read_csv(RUTA_DATOS)
df["created_at"] = pd.to_datetime(df["created_at"])
df.info()
df.head()


## 1. Preparación de la variable objetivo

Se conservan únicamente los dos grupos con suficientes observaciones para entrenar y evaluar el modelo. Las categorías casi vacías identificadas en la práctica de pruebas estadísticas no son adecuadas para una clasificación estable.

In [ ]:
def clasificar_interaccion(fila):
    if bool(fila["rating_valid"]) and bool(fila["has_comment_text"]):
        return "Calificación y texto"
    if (not bool(fila["rating_valid"])) and bool(fila["has_comment_text"]):
        return "Solo texto"
    return np.nan

df["categoria_interaccion"] = df.apply(clasificar_interaccion, axis=1)
clasificacion = df.dropna(subset=["categoria_interaccion"]).copy()
clasificacion["categoria_interaccion"].value_counts()


In [ ]:
plt.figure(figsize=(7, 4))
conteo_clases = clasificacion["categoria_interaccion"].value_counts()
plt.bar(conteo_clases.index, conteo_clases.values, color=["steelblue", "darkorange"])
plt.title("Distribución de las clases")
plt.xlabel("Categoría de interacción")
plt.ylabel("Número de comentarios")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 2. Selección de características

Se utilizan variables numéricas de interacción y reputación: `user_reputation`, `reply_count`, `thumbs_up`, `thumbs_down` y `best_score`. Se excluyen identificadores, texto libre, fechas y `stars` para evitar ruido o fuga de información.

KNN depende de las distancias, por lo que las características se estandarizarán dentro de un `Pipeline`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

caracteristicas = ["user_reputation", "reply_count", "thumbs_up", "thumbs_down", "best_score"]
X = clasificacion[caracteristicas].fillna(0)
y = clasificacion["categoria_interaccion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo_knn = Pipeline([
    ("escalado", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance"))
])
modelo_knn.fit(X_train, y_train)
predicciones = modelo_knn.predict(X_test)


In [ ]:
print(f"Accuracy: {accuracy_score(y_test, predicciones):.4f}")
print("\nReporte de clasificación:")
print(classification_report(y_test, predicciones, zero_division=0))


In [ ]:
matriz = confusion_matrix(y_test, predicciones, labels=modelo_knn.classes_)
display_confusion = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=modelo_knn.classes_)
display_confusion.plot(cmap="Blues", xticks_rotation=15)
plt.title("Matriz de confusión - KNN")
plt.tight_layout()
plt.show()


## 3. Selección del número de vecinos

Se comparan varios valores de `k`. La métrica principal será **F1 macro**, porque considera por igual a las clases y evita que la clase mayoritaria oculte un bajo desempeño en la clase minoritaria.

In [ ]:
from sklearn.metrics import f1_score

resultados_k = []
for k in range(1, 21):
    candidato = Pipeline([
        ("escalado", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance"))
    ])
    candidato.fit(X_train, y_train)
    pred_k = candidato.predict(X_test)
    resultados_k.append({
        "k": k,
        "accuracy": accuracy_score(y_test, pred_k),
        "f1_macro": f1_score(y_test, pred_k, average="macro")
    })

resultados_k = pd.DataFrame(resultados_k)
resultados_k.sort_values("f1_macro", ascending=False).head()


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultados_k["k"], resultados_k["accuracy"], marker="o", label="Accuracy")
plt.plot(resultados_k["k"], resultados_k["f1_macro"], marker="o", label="F1 macro")
plt.xlabel("Número de vecinos (k)")
plt.ylabel("Métrica")
plt.title("Comparación de valores de k")
plt.xticks(range(1, 21))
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Conclusiones

- El modelo se evalúa con datos no utilizados durante el entrenamiento.
- La estandarización es necesaria porque KNN calcula distancias entre observaciones.
- La matriz de confusión y el F1 macro permiten revisar si el desempeño es equilibrado entre `Calificación y texto` y `Solo texto`.
- El mejor valor de `k` debe seleccionarse con base en `resultados_k`, no únicamente por accuracy.
- Este modelo describe una relación predictiva en las variables disponibles; no implica causalidad.